Dataset Download

In [1]:
!pip install opencv-python matplotlib lxml
!wget https://zenodo.org/api/records/5106795/files-archive
!unzip -o files-archive
!unzip -o validation_set
!unzip -o train_set
!unzip -o test_set
!rm /kaggle/working/*.zip
!rm /kaggle/working/files-archive

--2026-03-12 10:19:31--  https://zenodo.org/api/records/5106795/files-archive
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 188.184.103.118, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘files-archive’

files-archive           [   <=>              ]   5.63G  23.8MB/s    in 4m 6s   

2026-03-12 10:23:38 (23.4 MB/s) - ‘files-archive’ saved [6043324001]

Archive:  files-archive
 extracting: validation_set.zip      
 extracting: test_set.zip            
 extracting: train_set.zip           
Archive:  validation_set.zip
  inflating: validation_set/IMG_1822_01.JPG  
  inflating: validation_set/IMG_1822_01.xml  
  inflating: validation_set/IMG_1822_02.JPG  
  inflating: validation_set/IMG_1822_02.xml  
  inflating: validation_set/IMG_1822_04.JPG  
  inflating: validation_set/IMG_1822_04.xml  
  inflating: validation_set/IMG_1822_14.JPG  
  inflati

Dataset -> Yolo Dataset

In [2]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

# Root from your screenshot
data_root = Path("/kaggle/working")

CLASS_NAMES = [
    "CYPRO","CYPRO_max","CYPRO_min","ECHCG","ECHCG_V1","ECHCG_V2",
    "ECHCG_Ve","NC","OE","POROL","SOLNI","SOLNI_V1","SOLNI_V2",
    "SOLNI_Vc","ZEAMX","ZEAMX_V1","ZEAMX_V3","ZEAMX_V4"
]

def count_xml_labels(folder_path):
    counts = Counter()
    # Find all XML files in the folder (including subfolders just in case)
    xml_files = list(folder_path.rglob("*.xml"))

    for xml_file in xml_files:
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()

            # Find every <object> tag in the XML
            for obj in root.findall('object'):
                name = obj.find('name').text
                if name in CLASS_NAMES:
                    counts[name] += 1
                else:
                    # In case there's a typo in the XML names vs your list
                    counts[name] += 1
        except Exception as e:
            continue # Skip corrupted files
    return counts

# Mapping based on your screenshot
splits = {
    "Train": data_root / "train_set",
    "Val":   data_root / "validation_set",
    "Test":  data_root / "test_set"
}

# Run the counter
results = {split_name: count_xml_labels(path) for split_name, path in splits.items()}

# Print Results
header = f"{'Class':<15} {'Train':>8} {'Val':>8} {'Test':>8}"
print(header)
print("-" * len(header))

for name in CLASS_NAMES:
    t = results["Train"].get(name, 0)
    v = results["Val"].get(name, 0)
    te = results["Test"].get(name, 0)
    print(f"{name:<15} {t:>8} {v:>8} {te:>8}")

Class              Train      Val     Test
------------------------------------------
CYPRO               6726     1977     3399
CYPRO_max           9183     2764     4860
CYPRO_min           7737     2385     4063
ECHCG               3160     1080     1293
ECHCG_V1               0        0       19
ECHCG_V2               0        0      205
ECHCG_Ve            2998      861     1545
NC                  6178     1912     3125
OE                     0        0       15
POROL                  0        0        5
SOLNI                  0        0        1
SOLNI_V1            4890     1614     2267
SOLNI_V2               0        0        2
SOLNI_Vc            7762     2288     3852
ZEAMX                 89       23       57
ZEAMX_V1           13014     3853     6309
ZEAMX_V3            4515     1310     2228
ZEAMX_V4            1522      483      746


STRATIFYING LOGIC - DO NOT TOUCH

In [3]:
"""
WeedMaize — Robust Multi-Label Stratified Dataset Splitter
===========================================================
Problem solved:
  - Dominant-label stratification ignores minority classes inside images
  - Pre-split folders carry historical bias
  - Rare classes randomly fall into only train or only test

Solution:
  - Iterative multi-label stratification (Sechidis et al., 2011)
    via skmultilearn — every class in every image is balanced
  - Fallback: guaranteed seeding of rare classes into ALL splits
    before stratification runs
  - Class weights computed and exported for use in your loss function
"""

import os
import shutil
import json
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

# ── Optional imports (graceful fallback) ────────────────────────────────────
try:
    from skmultilearn.model_selection import IterativeStratification
    HAS_SKMULTILEARN = True
except ImportError:
    HAS_SKMULTILEARN = False
    print("⚠️  skmultilearn not found — using fallback multi-label stratifier.")
    print("    Install with: pip install scikit-multilearn")

# ═══════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

ROOT       = Path("/kaggle/working")          # train_set / val_set / test_set live here
NEW_ROOT   = Path("/kaggle/working/stratified_dataset")  # output goes here

# Split ratios — must sum to 1.0
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

RANDOM_SEED = 42

# Minimum samples a class must have in EACH split after stratification.
# Images containing only ultra-rare classes are force-seeded first.
MIN_PER_SPLIT = 5

# ── Label merge map ─────────────────────────────────────────────────────────
# Classes absent from this map are DROPPED silently.
LABEL_MAP = {
    "CYPRO":     "CYPRO",
    "CYPRO_max": "CYPRO_max",
    "CYPRO_min": "CYPRO_min",

    "ECHCG":    "ECHCG",
    "ECHCG_V1": "ECHCG_Ve",   # merge V1 + V2 into Ve
    "ECHCG_V2": "ECHCG_Ve",
    "ECHCG_Ve": "ECHCG_Ve",

    "NC": "NC",

    "SOLNI":    "SOLNI_V1",   # merge base + V1 + V2
    "SOLNI_V1": "SOLNI_V1",
    "SOLNI_V2": "SOLNI_V1",
    "SOLNI_Vc": "SOLNI_Vc",   # keep separate — very numerous

    "ZEAMX":    "ZEAMX_V1",   # merge base into V1
    "ZEAMX_V1": "ZEAMX_V1",
    "ZEAMX_V3": "ZEAMX_V3",
    "ZEAMX_V4": "ZEAMX_V4",

    # OE, POROL — intentionally excluded
}

FINAL_CLASSES = sorted(set(LABEL_MAP.values()))
CLS_TO_IDX    = {c: i for i, c in enumerate(FINAL_CLASSES)}
N_CLASSES     = len(FINAL_CLASSES)


# ═══════════════════════════════════════════════════════════════════════════
# 2. COLLECT ALL IMAGE-ANNOTATION PAIRS
# ═══════════════════════════════════════════════════════════════════════════

def collect_all_pairs() -> list[dict]:
    """Scan all three original folders, parse XML, map labels."""
    all_data, skipped = [], 0
    print("\n── Scanning original dataset folders ──────────────────────────────")

    for folder in ["train_set", "validation_set", "test_set"]:
        folder_path = ROOT / folder
        if not folder_path.exists():
            print(f"  ⚠️  Not found, skipping: {folder_path}")
            continue

        xmls = sorted(folder_path.glob("*.xml"))
        print(f"  {folder}: {len(xmls)} XML files found")

        for xml_file in xmls:
            img_file = xml_file.with_suffix(".jpg")
            if not img_file.exists():
                img_file = xml_file.with_suffix(".JPG")
            if not img_file.exists():
                skipped += 1
                continue

            try:
                tree    = ET.parse(xml_file)
                root_el = tree.getroot()

                # All mapped labels in this image (one per bounding box)
                mapped = [
                    LABEL_MAP[obj.find("name").text]
                    for obj in root_el.findall("object")
                    if obj.find("name") is not None
                    and obj.find("name").text in LABEL_MAP
                ]
                if not mapped:
                    skipped += 1
                    continue

                # Multi-hot vector — presence of each class in this image
                multi_hot = np.zeros(N_CLASSES, dtype=np.int8)
                for lbl in mapped:
                    multi_hot[CLS_TO_IDX[lbl]] = 1

                # Box-level counts per class (used for class weights later)
                box_counts = Counter(mapped)

                all_data.append({
                    "xml":       xml_file,
                    "img":       img_file,
                    "stem":      f"{folder}__{xml_file.stem}",
                    "multi_hot": multi_hot,          # shape (N_CLASSES,)
                    "box_counts": box_counts,
                })

            except Exception as e:
                print(f"  ⚠️  Parse error {xml_file.name}: {e}")
                skipped += 1

    print(f"\n✅  Collected {len(all_data)} valid pairs. Skipped {skipped}.")
    return all_data


# ═══════════════════════════════════════════════════════════════════════════
# 3. PRINT CLASS DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════════════

def print_distribution(all_data: list[dict], title: str = "Full dataset"):
    """Show image-level and box-level counts per class."""
    img_counts = Counter()
    box_counts  = Counter()
    for d in all_data:
        for cls, idx in CLS_TO_IDX.items():
            if d["multi_hot"][idx]:
                img_counts[cls] += 1
        for cls, cnt in d["box_counts"].items():
            box_counts[cls] += cnt

    total_imgs = len(all_data)
    total_boxes = sum(box_counts.values())

    print(f"\n{'─'*70}")
    print(f"  {title}  ({total_imgs} images | {total_boxes} bounding boxes)")
    print(f"{'─'*70}")
    print(f"  {'Class':<15} {'Images':>8} {'Img%':>7}  {'Boxes':>8} {'Box%':>7}")
    print(f"  {'─'*15} {'─'*8} {'─'*7}  {'─'*8} {'─'*7}")
    for cls in FINAL_CLASSES:
        ic = img_counts.get(cls, 0)
        bc = box_counts.get(cls, 0)
        flag = "  ⚠️ RARE" if ic < MIN_PER_SPLIT * 3 else ""
        print(f"  {cls:<15} {ic:>8} {ic/total_imgs*100:>6.1f}%  "
              f"{bc:>8} {bc/total_boxes*100:>6.1f}%{flag}")
    print(f"{'─'*70}")
    return img_counts, box_counts


# ═══════════════════════════════════════════════════════════════════════════
# 4. RARE-CLASS FORCE SEEDING
# ═══════════════════════════════════════════════════════════════════════════

def force_seed_rare_classes(
    all_data: list[dict],
    img_counts: Counter,
) -> tuple[list[int], list[int], list[int], list[int]]:
    """
    For any class with fewer than MIN_PER_SPLIT * 3 images, guarantee
    at least MIN_PER_SPLIT samples land in each split by reserving them
    before stratification runs.

    Returns: (train_seed, val_seed, test_seed, remaining_indices)
    """
    rare_classes = [
        cls for cls in FINAL_CLASSES
        if img_counts.get(cls, 0) < MIN_PER_SPLIT * 3
    ]

    if not rare_classes:
        print("\n✅  No rare classes detected — skipping force-seed step.")
        return [], [], [], list(range(len(all_data)))

    print(f"\n⚠️  Rare classes detected (< {MIN_PER_SPLIT * 3} images): {rare_classes}")
    print("    Force-seeding these across all splits before stratification...")

    train_seed, val_seed, test_seed = [], [], []
    seeded = set()

    for cls in rare_classes:
        cls_idx = CLS_TO_IDX[cls]
        # Indices of images that contain this rare class
        candidates = [
            i for i, d in enumerate(all_data)
            if d["multi_hot"][cls_idx] and i not in seeded
        ]
        np.random.shuffle(candidates)  # reproducible via seed set before call

        n = len(candidates)
        # Proportional seeding respecting ratios
        n_train = max(MIN_PER_SPLIT, round(n * TRAIN_RATIO))
        n_val   = max(1, round(n * VAL_RATIO))
        n_test  = max(1, n - n_train - n_val)
        # Clip if not enough
        n_train = min(n_train, n - 2)
        n_val   = min(n_val,   n - n_train - 1)
        n_test  = n - n_train - n_val

        for i in candidates[:n_train]:
            train_seed.append(i); seeded.add(i)
        for i in candidates[n_train:n_train + n_val]:
            val_seed.append(i);   seeded.add(i)
        for i in candidates[n_train + n_val:n_train + n_val + n_test]:
            test_seed.append(i);  seeded.add(i)

        actual_seeded = candidates[:n_train + n_val + n_test]
        print(f"    {cls}: seeded {n_train} train | {n_val} val | {n_test} test "
              f"(of {n} total)")

    remaining = [i for i in range(len(all_data)) if i not in seeded]
    print(f"\n  Seeded {len(seeded)} images. {len(remaining)} remain for stratification.")
    return train_seed, val_seed, test_seed, remaining


# ═══════════════════════════════════════════════════════════════════════════
# 5A. ITERATIVE MULTI-LABEL STRATIFICATION (preferred)
# ═══════════════════════════════════════════════════════════════════════════

def iterative_stratify(
    indices: list[int],
    all_data: list[dict],
) -> tuple[list[int], list[int], list[int]]:
    """Use skmultilearn IterativeStratification on remaining pool."""
    X = np.array(indices).reshape(-1, 1)
    Y = np.vstack([all_data[i]["multi_hot"] for i in indices])

    # Two-step: split off test, then split remainder into train/val
    test_size  = TEST_RATIO / (TRAIN_RATIO + VAL_RATIO + TEST_RATIO)
    val_size_2 = VAL_RATIO  / (TRAIN_RATIO + VAL_RATIO)

    # Pre-shuffle for reproducibility — IterativeStratification does not
    # accept random_state when shuffle=False (its only supported mode)
    rng = np.random.default_rng(RANDOM_SEED)
    perm = rng.permutation(len(indices))
    X, Y = X[perm], Y[perm]

    strat1 = IterativeStratification(
        n_splits=2,
        order=2,
        sample_distribution_per_fold=[test_size, 1 - test_size],
    )
    for train_val_pos, test_pos in strat1.split(X, Y):
        break  # only need first fold

    X_tv, Y_tv = X[train_val_pos], Y[train_val_pos]

    strat2 = IterativeStratification(
        n_splits=2,
        order=2,
        sample_distribution_per_fold=[val_size_2, 1 - val_size_2],
    )
    for train_pos, val_pos in strat2.split(X_tv, Y_tv):
        break

    train_idx = [X_tv[i, 0] for i in train_pos]
    val_idx   = [X_tv[i, 0] for i in val_pos]
    test_idx  = [X[i, 0] for i in test_pos]

    return train_idx, val_idx, test_idx


# ═══════════════════════════════════════════════════════════════════════════
# 5B. FALLBACK: GREEDY MULTI-LABEL STRATIFICATION
# ═══════════════════════════════════════════════════════════════════════════

def greedy_multilabel_stratify(
    indices: list[int],
    all_data: list[dict],
) -> tuple[list[int], list[int], list[int]]:
    """
    Greedy fallback when skmultilearn is unavailable.
    Assigns each image to the split where its rarest class is most under-represented.
    Substantially better than dominant-label single-label stratification.
    """
    rng = np.random.default_rng(RANDOM_SEED)
    idx_arr = np.array(indices)
    rng.shuffle(idx_arr)

    # Track how many images of each class are in each split
    cls_counts = {
        "train": np.zeros(N_CLASSES, dtype=int),
        "val":   np.zeros(N_CLASSES, dtype=int),
        "test":  np.zeros(N_CLASSES, dtype=int),
    }
    split_total = {"train": 0, "val": 0, "test": 0}
    split_map   = {"train": [], "val": [], "test": []}

    target = {
        "train": TRAIN_RATIO,
        "val":   VAL_RATIO,
        "test":  TEST_RATIO,
    }
    n_total = len(idx_arr)

    for i in idx_arr:
        hot    = all_data[i]["multi_hot"]
        active = np.where(hot)[0]   # classes present in this image

        best_split, best_score = None, float("inf")
        for split_name in ["train", "val", "test"]:
            # Penalise if this split is already over its target ratio
            size_penalty = split_total[split_name] / max(n_total * target[split_name], 1)

            # For each active class, compute how under-represented it is here
            deficit = 0.0
            for ci in active:
                total_of_cls = sum(cls_counts[s][ci] for s in ["train", "val", "test"])
                if total_of_cls == 0:
                    deficit += 1.0   # unseen class — high priority
                else:
                    expected = target[split_name] * total_of_cls
                    actual   = cls_counts[split_name][ci]
                    deficit += max(0.0, expected - actual) / total_of_cls

            # Lower score = better (more deficit, less size penalty)
            score = size_penalty - deficit / max(len(active), 1)

            if score < best_score:
                best_score, best_split = score, split_name

        split_map[best_split].append(i)
        split_total[best_split] += 1
        for ci in active:
            cls_counts[best_split][ci] += 1

    return split_map["train"], split_map["val"], split_map["test"]


# ═══════════════════════════════════════════════════════════════════════════
# 6. COMPUTE CLASS WEIGHTS (for loss balancing during training)
# ═══════════════════════════════════════════════════════════════════════════

def compute_class_weights(train_indices: list[int], all_data: list[dict]) -> dict:
    """
    Inverse-frequency weights on BOUNDING BOX counts in the training set.
    Export as JSON for direct use in your training script.
    """
    box_counts = Counter()
    for i in train_indices:
        for cls, cnt in all_data[i]["box_counts"].items():
            box_counts[cls] += cnt

    total = sum(box_counts.values())
    n_cls = len(FINAL_CLASSES)

    # Balanced inverse-frequency: w_i = total / (n_classes * count_i)
    weights = {}
    for cls in FINAL_CLASSES:
        cnt = box_counts.get(cls, 0)
        if cnt == 0:
            weights[cls] = 0.0   # shouldn't happen after seeding
        else:
            weights[cls] = round(total / (n_cls * cnt), 4)

    return weights


# ═══════════════════════════════════════════════════════════════════════════
# 7. COPY / SYMLINK FILES
# ═══════════════════════════════════════════════════════════════════════════

def create_split_files(
    indices: list[int],
    all_data: list[dict],
    split_name: str,
    use_symlinks: bool = True,
):
    """Write image + XML into split folder (symlink or hard copy)."""
    dest = NEW_ROOT / split_name
    dest.mkdir(parents=True, exist_ok=True)

    for i in indices:
        item = all_data[i]
        xml_dst = dest / f"{item['stem']}.xml"
        img_dst = dest / f"{item['stem']}{item['img'].suffix}"

        for src, dst in [(item["xml"], xml_dst), (item["img"], img_dst)]:
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            if use_symlinks:
                os.symlink(src.resolve(), dst)
            else:
                shutil.copy2(src, dst)

    print(f"  ✅ {split_name:<6}: {len(indices)} images → {dest}")


# ═══════════════════════════════════════════════════════════════════════════
# 8. SPLIT VALIDATION — catch mismatches BEFORE training
# ═══════════════════════════════════════════════════════════════════════════

def validate_splits(
    train_idx: list[int],
    val_idx:   list[int],
    test_idx:  list[int],
    all_data:  list[dict],
) -> bool:
    """
    Assert:
      1. No image appears in more than one split
      2. Every class present in test/val also exists in train
      3. No split has 0 images of any class that exists in the full dataset
    """
    print("\n── Validating splits ───────────────────────────────────────────────")
    ok = True

    # (1) Overlap check
    sets = [set(train_idx), set(val_idx), set(test_idx)]
    for a, b, name in [(sets[0], sets[1], "train∩val"),
                       (sets[0], sets[2], "train∩test"),
                       (sets[1], sets[2], "val∩test")]:
        overlap = a & b
        if overlap:
            print(f"  ❌ OVERLAP in {name}: {len(overlap)} images shared!")
            ok = False
        else:
            print(f"  ✅ No overlap: {name}")

    # (2 & 3) Class coverage
    def cls_set(idxs):
        s = set()
        for i in idxs:
            s.update(c for c, v in zip(FINAL_CLASSES, all_data[i]["multi_hot"]) if v)
        return s

    train_cls = cls_set(train_idx)
    val_cls   = cls_set(val_idx)
    test_cls  = cls_set(test_idx)
    all_cls   = cls_set(list(range(len(all_data))))

    missing_val  = all_cls - val_cls
    missing_test = all_cls - test_cls
    missing_train_but_in_test = (test_cls | val_cls) - train_cls

    if missing_val:
        print(f"  ⚠️  Classes missing from VAL:  {missing_val}")
    else:
        print(f"  ✅ All classes present in val")

    if missing_test:
        print(f"  ⚠️  Classes missing from TEST: {missing_test}")
    else:
        print(f"  ✅ All classes present in test")

    if missing_train_but_in_test:
        print(f"  ❌ CRITICAL: Classes in test/val but NOT in train: "
              f"{missing_train_but_in_test}")
        ok = False
    else:
        print(f"  ✅ Train contains all classes seen in val/test")

    print(f"{'─'*70}")
    return ok


# ═══════════════════════════════════════════════════════════════════════════
# 9. MAIN ORCHESTRATION
# ═══════════════════════════════════════════════════════════════════════════

def main():
    np.random.seed(RANDOM_SEED)

    # ── Step 1: Collect data ──────────────────────────────────────────────
    all_data = collect_all_pairs()
    if not all_data:
        raise RuntimeError("No data found — check ROOT path.")

    # ── Step 2: Show full distribution ───────────────────────────────────
    img_counts, box_counts = print_distribution(all_data, "Full merged dataset")

    # ── Step 3: Force-seed rare classes ──────────────────────────────────
    train_seed, val_seed, test_seed, remaining = force_seed_rare_classes(
        all_data, img_counts
    )

    # ── Step 4: Stratify remaining pool ──────────────────────────────────
    if len(remaining) == 0:
        # Edge case: entire dataset was rare-seeded
        train_r, val_r, test_r = [], [], []
    elif HAS_SKMULTILEARN:
        print("\n── Running IterativeStratification (skmultilearn) ──────────────────")
        train_r, val_r, test_r = iterative_stratify(remaining, all_data)
    else:
        print("\n── Running greedy multi-label stratification (fallback) ─────────────")
        train_r, val_r, test_r = greedy_multilabel_stratify(remaining, all_data)

    # ── Step 5: Merge seed + stratified pools ────────────────────────────
    train_idx = train_seed + list(train_r)
    val_idx   = val_seed   + list(val_r)
    test_idx  = test_seed  + list(test_r)

    # ── Step 6: Validate ─────────────────────────────────────────────────
    valid = validate_splits(train_idx, val_idx, test_idx, all_data)
    if not valid:
        raise RuntimeError(
            "Split validation FAILED — fix class imbalance before proceeding."
        )

    # ── Step 7: Per-split distribution reports ───────────────────────────
    for name, idxs in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
        print_distribution([all_data[i] for i in idxs], f"{name} split")

    # ── Step 8: Compute & export class weights ───────────────────────────
    weights = compute_class_weights(train_idx, all_data)
    print("\n── Class weights for loss balancing (based on train box counts) ───")
    print(f"  {'Class':<15} {'Weight':>8}")
    for cls, w in sorted(weights.items(), key=lambda x: -x[1]):
        print(f"  {cls:<15} {w:>8.4f}")

    weights_path = NEW_ROOT / "class_weights.json"
    NEW_ROOT.mkdir(parents=True, exist_ok=True)
    with open(weights_path, "w") as f:
        json.dump({"class_weights": weights, "classes": FINAL_CLASSES}, f, indent=2)
    print(f"\n  Saved class weights → {weights_path}")

    # ── Step 9: Write split summary JSON ─────────────────────────────────
    summary = {
        "total":  len(all_data),
        "train":  len(train_idx),
        "val":    len(val_idx),
        "test":   len(test_idx),
        "ratios": {
            "train": round(len(train_idx) / len(all_data), 3),
            "val":   round(len(val_idx)   / len(all_data), 3),
            "test":  round(len(test_idx)  / len(all_data), 3),
        },
        "classes": FINAL_CLASSES,
        "stratification": "iterative" if HAS_SKMULTILEARN else "greedy_multilabel",
    }
    with open(NEW_ROOT / "split_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    # ── Step 10: Write files ──────────────────────────────────────────────
    print("\n── Writing split directories ───────────────────────────────────────")
    create_split_files(train_idx, all_data, "train")
    create_split_files(val_idx,   all_data, "val")
    create_split_files(test_idx,  all_data, "test")

    # ── Final summary ─────────────────────────────────────────────────────
    print(f"""
╔══════════════════════════════════════════════════════════════╗
║                      SPLIT COMPLETE ✅                       ║
╠══════════════════════════════════════════════════════════════╣
║  Total images : {len(all_data):<7}                                  ║
║  Train        : {len(train_idx):<7} ({len(train_idx)/len(all_data)*100:.1f}%)                           ║
║  Val          : {len(val_idx):<7} ({len(val_idx)/len(all_data)*100:.1f}%)                            ║
║  Test         : {len(test_idx):<7} ({len(test_idx)/len(all_data)*100:.1f}%)                            ║
║  Classes      : {N_CLASSES:<7}                                  ║
║  Strategy     : {'iterative' if HAS_SKMULTILEARN else 'greedy_multilabel':<20}                   ║
╚══════════════════════════════════════════════════════════════╝

📂 Output  : {NEW_ROOT}
⚖️  Weights : {weights_path}

Next step — use class weights in your training loss, e.g.:
  weights = json.load(open('class_weights.json'))['class_weights']
  # PyTorch:
  weight_tensor = torch.tensor([weights[c] for c in FINAL_CLASSES])
  criterion = FocalLoss(alpha=weight_tensor, gamma=2.0)
""")


if __name__ == "__main__":
    main()


── Scanning original dataset folders ──────────────────────────────
  train_set: 4368 XML files found
  validation_set: 1310 XML files found
  test_set: 2184 XML files found

✅  Collected 7862 valid pairs. Skipped 0.

──────────────────────────────────────────────────────────────────────
  Full merged dataset  (7862 images | 122295 bounding boxes)
──────────────────────────────────────────────────────────────────────
  Class             Images    Img%     Boxes    Box%
  ─────────────── ──────── ───────  ──────── ───────
  CYPRO               5335   67.9%     12102    9.9%
  CYPRO_max           5711   72.6%     16807   13.7%
  CYPRO_min           5395   68.6%     14185   11.6%
  ECHCG               1475   18.8%      5533    4.5%
  ECHCG_Ve            2433   30.9%      5628    4.6%
  NC                  4431   56.4%     11215    9.2%
  SOLNI_V1            1268   16.1%      8774    7.2%
  SOLNI_Vc            3993   50.8%     13902   11.4%
  ZEAMX_V1            5408   68.8%     23345   1

MODEL TRAINING LOGIC, CHANGE THE CELL BELOW

In [4]:
"""
WeedMaize — YOLO11n Feature Distillation Training Pipeline
=========================================================
Method 2 implemented here:
  - teacher: trained YOLO11m checkpoint
  - student: YOLO11n
  - feature KD: attention transfer on the 3 FPN maps entering the Detect head
  - logit KD: temperature-scaled MSE on raw detection head outputs

Notes:
  - This notebook version uses single-GPU training intentionally because
    teacher + student forward passes roughly double VRAM usage.
  - YOLO labels are rebuilt with the same merge map used in the stratification
    step, so merged classes stay consistent during student training.
"""
!pip install ultralytics
import json
import os
import shutil
import time
import warnings
import xml.etree.ElementTree as ET
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.utils import LOGGER, RANK, TQDM, colorstr
from ultralytics.utils.torch_utils import autocast, unset_deterministic, unwrap_model

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_DIR = Path("/kaggle/working/stratified_dataset")
TRAIN_DIR = BASE_DIR / "train"
VAL_DIR = BASE_DIR / "val"
TEST_DIR = BASE_DIR / "test"
YOLO_DIR = BASE_DIR / "weed_yolo"
WEIGHTS_F = BASE_DIR / "class_weights.json"

LABEL_MAP = {
    "CYPRO": "CYPRO",
    "CYPRO_max": "CYPRO_max",
    "CYPRO_min": "CYPRO_min",
    "ECHCG": "ECHCG",
    "ECHCG_V1": "ECHCG_Ve",
    "ECHCG_V2": "ECHCG_Ve",
    "ECHCG_Ve": "ECHCG_Ve",
    "NC": "NC",
    "SOLNI": "SOLNI_V1",
    "SOLNI_V1": "SOLNI_V1",
    "SOLNI_V2": "SOLNI_V1",
    "SOLNI_Vc": "SOLNI_Vc",
    "ZEAMX": "ZEAMX_V1",
    "ZEAMX_V1": "ZEAMX_V1",
    "ZEAMX_V3": "ZEAMX_V3",
    "ZEAMX_V4": "ZEAMX_V4",
}

CLASSES = [
    "CYPRO",
    "CYPRO_max",
    "CYPRO_min",
    "ECHCG",
    "ECHCG_Ve",
    "NC",
    "SOLNI_V1",
    "SOLNI_Vc",
    "ZEAMX_V1",
    "ZEAMX_V3",
    "ZEAMX_V4",
]
CLASS_MAP = {name: idx for idx, name in enumerate(CLASSES)}
SPLITS = {
    "train": TRAIN_DIR,
    "val": VAL_DIR,
    "test": TEST_DIR,
}

TEACHER_CANDIDATES = [
    Path("/kaggle/input/models/kushagraistaken/teacher-model-final/pytorch/default/1/best.pt"),
    Path("/kaggle/working/yolo_results/best_model.pt"),
    Path("/kaggle/working/runs/weedmaize_yolo11m/weights/best.pt"),
    Path("/kaggle/working/teacher_export/models/best.pt"),
    Path("/kaggle/working/best.pt"),
]


def convert_box(img_w, img_h, xmin, ymin, xmax, ymax):
    """Convert Pascal VOC absolute coords to YOLO normalized coords."""
    cx = ((xmin + xmax) / 2.0) / img_w
    cy = ((ymin + ymax) / 2.0) / img_h
    bw = (xmax - xmin) / img_w
    bh = (ymax - ymin) / img_h
    return cx, cy, bw, bh



def prepare_yolo_dataset():
    for split in SPLITS:
        (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    total_images, total_boxes, skipped_boxes = 0, 0, 0

    for split, src_dir in SPLITS.items():
        img_dst = YOLO_DIR / "images" / split
        lbl_dst = YOLO_DIR / "labels" / split
        split_images, split_boxes = 0, 0

        image_files = []
        for pattern in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
            image_files.extend(sorted(src_dir.glob(pattern)))

        for img_path in image_files:
            xml_path = src_dir / f"{img_path.stem}.xml"
            dst_img = img_dst / img_path.name
            shutil.copy2(img_path.resolve(), dst_img)
            split_images += 1

            if not xml_path.exists():
                continue

            tree = ET.parse(xml_path)
            root_el = tree.getroot()
            size = root_el.find("size")
            img_w = int(size.find("width").text)
            img_h = int(size.find("height").text)

            label_lines = []
            for obj in root_el.findall("object"):
                raw_name = obj.findtext("name")
                mapped_name = LABEL_MAP.get(raw_name)
                if mapped_name is None or mapped_name not in CLASS_MAP:
                    skipped_boxes += 1
                    continue

                box = obj.find("bndbox")
                xmin = float(box.find("xmin").text)
                ymin = float(box.find("ymin").text)
                xmax = float(box.find("xmax").text)
                ymax = float(box.find("ymax").text)
                cx, cy, bw, bh = convert_box(img_w, img_h, xmin, ymin, xmax, ymax)
                label_lines.append(
                    f"{CLASS_MAP[mapped_name]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
                )
                split_boxes += 1

            if label_lines:
                (lbl_dst / f"{img_path.stem}.txt").write_text(
                    "\n".join(label_lines) + "\n",
                    encoding="utf-8",
                )

        total_images += split_images
        total_boxes += split_boxes
        print(f"  {split:<6}: {split_images} images | {split_boxes} boxes")

    yaml_lines = [
        f"path: {YOLO_DIR}",
        "train: images/train",
        "val: images/val",
        "test: images/test",
        "",
        f"nc: {len(CLASSES)}",
        "",
        "names:",
    ]
    for idx, name in enumerate(CLASSES):
        yaml_lines.append(f"  {idx}: {name}")

    yaml_path = YOLO_DIR / "dataset.yaml"
    yaml_path.write_text("\n".join(yaml_lines) + "\n", encoding="utf-8")

    print(f"\n✅ YOLO dataset prepared at {YOLO_DIR}")
    print(f"✅ dataset.yaml written → {yaml_path}")
    print(f"✅ Total converted: {total_images} images | {total_boxes} boxes")
    if skipped_boxes:
        print(f"⚠️  {skipped_boxes} boxes skipped because their labels were intentionally dropped")

    return yaml_path



def find_teacher_weights():
    for path in TEACHER_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Teacher checkpoint not found. Place best.pt in one of these paths: "
        + ", ".join(str(p) for p in TEACHER_CANDIDATES)
    )



def collect_4d_tensors(payload):
    if torch.is_tensor(payload):
        return [payload] if payload.ndim == 4 else []
    if isinstance(payload, dict):
        tensors = []
        for value in payload.values():
            tensors.extend(collect_4d_tensors(value))
        return tensors
    if isinstance(payload, (list, tuple)):
        tensors = []
        for item in payload:
            tensors.extend(collect_4d_tensors(item))
        return tensors
    return []


class DetectInputTap:
    """Capture the feature pyramid maps that are fed into the Detect head."""

    def __init__(self, model):
        self.features = []
        self.handle = model.model[-1].register_forward_pre_hook(self._hook)

    def _hook(self, module, inputs):
        x = inputs[0] if inputs else []
        if isinstance(x, (list, tuple)):
            self.features = [tensor for tensor in x if torch.is_tensor(tensor)]
        elif torch.is_tensor(x):
            self.features = [x]
        else:
            self.features = []

    def close(self):
        self.handle.remove()



def attention_transfer_loss(student_features, teacher_features, device):
    student_levels = student_features[-3:]
    teacher_levels = teacher_features[-3:]
    if not student_levels or not teacher_levels:
        return torch.zeros((), device=device)

    losses = []
    for student_feat, teacher_feat in zip(student_levels, teacher_levels):
        if teacher_feat.shape[-2:] != student_feat.shape[-2:]:
            teacher_feat = F.interpolate(
                teacher_feat,
                size=student_feat.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        student_att = F.normalize(student_feat.pow(2).mean(1).flatten(1), p=2, dim=1)
        teacher_att = F.normalize(teacher_feat.pow(2).mean(1).flatten(1), p=2, dim=1)
        losses.append(F.mse_loss(student_att, teacher_att))

    return torch.stack(losses).mean() if losses else torch.zeros((), device=device)



def logit_distillation_loss(student_outputs, teacher_outputs, temperature, device):
    student_maps = collect_4d_tensors(student_outputs)[-3:]
    teacher_maps = collect_4d_tensors(teacher_outputs)[-3:]
    if not student_maps or not teacher_maps:
        return torch.zeros((), device=device)

    losses = []
    for student_map, teacher_map in zip(student_maps, teacher_maps):
        if teacher_map.shape[-2:] != student_map.shape[-2:]:
            teacher_map = F.interpolate(
                teacher_map,
                size=student_map.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        min_channels = min(student_map.shape[1], teacher_map.shape[1])
        student_map = student_map[:, :min_channels]
        teacher_map = teacher_map[:, :min_channels]
        losses.append(
            F.mse_loss(student_map / temperature, teacher_map / temperature)
            * (temperature ** 2)
        )

    return torch.stack(losses).mean() if losses else torch.zeros((), device=device)


class FeatureDistillationTrainer(DetectionTrainer):
    def __init__(
        self,
        *args,
        teacher_weights,
        kd_feat_lambda=1.0,
        kd_logit_lambda=0.25,
        kd_temperature=2.0,
        **kwargs,
    ):
        self.teacher_weights = Path(teacher_weights)
        self.kd_feat_lambda = float(kd_feat_lambda)
        self.kd_logit_lambda = float(kd_logit_lambda)
        self.kd_temperature = float(kd_temperature)
        self.loss_names = (
            "box_loss",
            "cls_loss",
            "dfl_loss",
            "feat_kd_loss",
            "logit_kd_loss",
        )
        super().__init__(*args, **kwargs)

    def get_validator(self):
        validator = super().get_validator()
        self.loss_names = (
            "box_loss",
            "cls_loss",
            "dfl_loss",
            "feat_kd_loss",
            "logit_kd_loss",
        )
        return validator

    def label_loss_items(self, loss_items=None, prefix="train"):
        keys = [f"{prefix}/{name}" for name in self.loss_names]
        if loss_items is None:
            return keys
        values = [round(float(x), 5) for x in loss_items]
        return dict(zip(keys, values))

    def progress_string(self):
        return ("\n" + "%11s" * (4 + len(self.loss_names))) % (
            "Epoch",
            "GPU_mem",
            *self.loss_names,
            "Instances",
            "Size",
        )

    def _set_teacher_mode(self):
        self.teacher.train()
        for module in self.teacher.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                module.eval()

    def _pad_validation_losses(self, metrics):
        for key in self.label_loss_items(prefix="val"):
            metrics.setdefault(key, 0.0)
        return metrics

    def validate(self):
        model = self.ema.ema if self.ema else unwrap_model(self.model)
        eval_model = deepcopy(model).to(self.device)
        eval_model.eval()
        self.validator.args.plots = self.args.plots
        self.validator.args.compile = False
        metrics = self.validator(model=eval_model)
        del eval_model
        self._clear_memory(threshold=0.5)

        if metrics is None:
            return None, None

        fitness = metrics.pop("fitness", -float(self.loss.detach().cpu()))
        self._pad_validation_losses(metrics)
        if not self.best_fitness or self.best_fitness < fitness:
            self.best_fitness = fitness
        return metrics, fitness

    def _setup_train(self):
        super()._setup_train()
        if self.world_size > 1:
            raise RuntimeError(
                "This KD notebook trainer supports single-GPU training only. Set device=0."
            )

        self.teacher = YOLO(str(self.teacher_weights)).model.to(self.device)
        for param in self.teacher.parameters():
            param.requires_grad = False

        self.teacher_tap = DetectInputTap(self.teacher)
        self.student_tap = DetectInputTap(unwrap_model(self.model))
        self._set_teacher_mode()
        LOGGER.info(f"Teacher weights loaded from {self.teacher_weights}")

    def _forward_with_kd(self, batch):
        student_model = unwrap_model(self.model)
        student_outputs = self.model(batch["img"])
        det_loss, det_items = student_model.loss(batch, student_outputs)

        with torch.no_grad():
            self._set_teacher_mode()
            teacher_outputs = self.teacher(batch["img"])

        feat_loss = attention_transfer_loss(
            self.student_tap.features,
            self.teacher_tap.features,
            device=det_items.device,
        )
        logit_loss = logit_distillation_loss(
            student_outputs,
            teacher_outputs,
            temperature=self.kd_temperature,
            device=det_items.device,
        )

        total_loss = det_loss + self.kd_feat_lambda * feat_loss + self.kd_logit_lambda * logit_loss
        extra_losses = torch.stack((feat_loss.detach(), logit_loss.detach()))
        loss_items = torch.cat((det_items.detach(), extra_losses))
        return total_loss, loss_items

    def _do_train(self):
        self._setup_train()
        nb = len(self.train_loader)
        nw = max(round(self.args.warmup_epochs * nb), 100) if self.args.warmup_epochs > 0 else -1
        last_opt_step = -1
        self.epoch_time_start = time.time()
        self.train_time_start = time.time()

        self.run_callbacks("on_train_start")
        LOGGER.info(
            f"Image sizes {self.args.imgsz} train, {self.args.imgsz} val\n"
            f"Using {self.train_loader.num_workers} dataloader workers\n"
            f"Logging results to {colorstr('bold', self.save_dir)}\n"
            f"Starting feature distillation training for {self.epochs} epochs..."
        )

        if self.args.close_mosaic:
            base_idx = (self.epochs - self.args.close_mosaic) * nb
            self.plot_idx.extend([base_idx, base_idx + 1, base_idx + 2])

        self.optimizer.zero_grad()

        for epoch in range(self.start_epoch, self.epochs):
            self.epoch = epoch
            self.run_callbacks("on_train_epoch_start")
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                self.scheduler.step()

            self._model_train()
            self._set_teacher_mode()

            if epoch == (self.epochs - self.args.close_mosaic):
                self._close_dataloader_mosaic()
                self.train_loader.reset()

            pbar = enumerate(self.train_loader)
            if RANK in {-1, 0}:
                LOGGER.info(self.progress_string())
                pbar = TQDM(enumerate(self.train_loader), total=nb)

            self.tloss = None
            for i, batch in pbar:
                self.run_callbacks("on_train_batch_start")
                ni = i + nb * epoch

                if ni <= nw:
                    xi = [0, nw]
                    self.accumulate = max(
                        1,
                        int(np.interp(ni, xi, [1, self.args.nbs / self.batch_size]).round()),
                    )
                    for group in self.optimizer.param_groups:
                        group["lr"] = np.interp(
                            ni,
                            xi,
                            [
                                self.args.warmup_bias_lr if group.get("param_group") == "bias" else 0.0,
                                group["initial_lr"] * self.lf(epoch),
                            ],
                        )
                        if "momentum" in group:
                            group["momentum"] = np.interp(
                                ni,
                                xi,
                                [self.args.warmup_momentum, self.args.momentum],
                            )

                with autocast(self.amp):
                    batch = self.preprocess_batch(batch)
                    total_loss, loss_items = self._forward_with_kd(batch)
                    self.loss = total_loss.sum()
                    self.loss_items = loss_items
                    self.tloss = loss_items if self.tloss is None else (self.tloss * i + loss_items) / (i + 1)

                self.scaler.scale(self.loss).backward()

                if ni - last_opt_step >= self.accumulate:
                    self.optimizer_step()
                    last_opt_step = ni

                if RANK in {-1, 0}:
                    loss_length = self.tloss.shape[0] if len(self.tloss.shape) else 1
                    pbar.set_description(
                        (("%11s" * 2) + ("%11.4g" * (2 + loss_length)))
                        % (
                            f"{epoch + 1}/{self.epochs}",
                            f"{self._get_memory():.3g}G",
                            *(self.tloss if loss_length > 1 else torch.unsqueeze(self.tloss, 0)),
                            batch["cls"].shape[0],
                            batch["img"].shape[-1],
                        )
                    )
                    self.run_callbacks("on_batch_end")
                    if self.args.plots and ni in self.plot_idx:
                        self.plot_training_samples(batch, ni)

                self.run_callbacks("on_train_batch_end")

            if hasattr(unwrap_model(self.model).criterion, "update"):
                unwrap_model(self.model).criterion.update()

            self.lr = {f"lr/pg{idx}": group["lr"] for idx, group in enumerate(self.optimizer.param_groups)}
            self.run_callbacks("on_train_epoch_end")

            if RANK in {-1, 0}:
                self.ema.update_attr(self.model, include=["yaml", "nc", "args", "names", "stride"])
                self._clear_memory(threshold=0.5)
                self.metrics, self.fitness = self.validate()
                self.save_metrics(metrics={**self.label_loss_items(self.tloss), **self.metrics, **self.lr})

                final_epoch = epoch + 1 >= self.epochs
                self.stop |= self.stopper(epoch + 1, self.fitness) or final_epoch
                if self.args.save or final_epoch:
                    self.save_model()
                    self.run_callbacks("on_model_save")

            self.run_callbacks("on_fit_epoch_end")
            self._clear_memory(0.5)
            if self.stop:
                break

        seconds = time.time() - self.train_time_start
        LOGGER.info(f"\n{epoch - self.start_epoch + 1} epochs completed in {seconds / 3600:.3f} hours.")
        self.final_eval()

        if RANK in {-1, 0}:
            if self.args.plots:
                self.plot_metrics()
            self.run_callbacks("on_train_end")

        self.teacher_tap.close()
        self.student_tap.close()
        self._clear_memory()
        unset_deterministic()
        self.run_callbacks("teardown")


print(f"Student classes ({len(CLASSES)}): {CLASSES}")
yaml_path = prepare_yolo_dataset()
teacher_weights = find_teacher_weights()
print(f"\n✅ Teacher checkpoint found → {teacher_weights}")

if WEIGHTS_F.exists():
    kd_class_weights = json.loads(WEIGHTS_F.read_text(encoding="utf-8"))
    print("⚖️  Training-set class weights available for reference.")
    print(kd_class_weights)

train_args = dict(
    model="yolo11n.pt",
    data=str(yaml_path),
    epochs=100,
    imgsz=832,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=4,
    cache="disk",
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    amp=True,
    patience=25,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.1,
    scale=0.6,
    shear=2,
    perspective=0.0005,
    fliplr=0.5,
    flipud=0.3,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.2,
    erasing=0.4,
    dropout=0.0,
    weight_decay=0.0005,
    close_mosaic=15,
    cls=0.5,
    box=7.5,
    dfl=1.5,
    project="/kaggle/working/runs",
    name="weedmaize_yolo11n_feature_kd",
    exist_ok=True,
    save=True,
    save_period=10,
    val=True,
    plots=True,
)

trainer = FeatureDistillationTrainer(
    overrides=train_args,
    teacher_weights=teacher_weights,
    kd_feat_lambda=1.0,
    kd_logit_lambda=0.25,
    kd_temperature=2.0,
)
trainer.train()

student_run_dir = Path(trainer.save_dir)
results_dst = Path("/kaggle/working/yolo_results")
results_dst.mkdir(exist_ok=True)
weights_dir = student_run_dir / "weights"

shutil.copy2(weights_dir / "best.pt", results_dst / "best_model.pt")
shutil.copy2(weights_dir / "last.pt", results_dst / "last_model.pt")

for artifact in student_run_dir.glob("*.png"):
    shutil.copy2(artifact, results_dst / artifact.name)
for artifact in student_run_dir.glob("*.csv"):
    shutil.copy2(artifact, results_dst / artifact.name)

feature_kd_config = {
    "teacher_model": str(teacher_weights),
    "student_model": "yolo11n.pt",
    "run_dir": str(student_run_dir),
    "dataset_yaml": str(yaml_path),
    "kd_feat_lambda": 1.0,
    "kd_logit_lambda": 0.25,
    "kd_temperature": 2.0,
    "feature_distillation": "attention_transfer_on_detect_fpn_inputs",
    "logit_distillation": "temperature_scaled_mse_on_raw_detect_outputs",
}
(results_dst / "feature_kd_config.json").write_text(
    json.dumps(feature_kd_config, indent=2),
    encoding="utf-8",
)

print("\n🎉 Feature distillation training complete!")
print(f"   Student run dir → {student_run_dir}")
print(f"   Best weights    → {results_dst / 'best_model.pt'}")
print(f"   Last weights    → {results_dst / 'last_model.pt'}")
print(f"   Training CSV    → {results_dst / 'results.csv'}")
print(f"   KD config       → {results_dst / 'feature_kd_config.json'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Student classes (11): ['CYPRO', 'CYPRO_max', 'CYPRO_min', 'ECHCG', 'ECHCG_Ve', 'NC', 'SOLNI_V1', 'SOLNI_Vc', 'ZEAMX_V1', 'ZEAMX_V3', 'ZEAMX_V4']
  train : 6284 images | 97052 boxes
  val   : 786 images | 12417 boxes
  test  : 792 images | 12826 boxes

✅ YOLO dataset prepared at /kaggle/working/stratified_dataset/weed_yolo
✅ dataset.yaml written → /kaggle/working/stratified_dataset/weed_yolo/dataset.yaml
✅ Total converted: 7862 images | 122295 boxes
⚠️  20 boxes skipped because their labels were intentionally dropped

✅ Teacher checkpoint found → /kaggle/input/models/kushagraistaken/teacher-model-final/pyt

MODEL EFFICIENCY BENCHMARK

In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# MODEL EFFICIENCY BENCHMARK (Params, GFLOPs, Latency, FPS, Size)
# ═══════════════════════════════════════════════════════════════════════════

import torch
import time
import pandas as pd
from thop import profile
from ultralytics import YOLO

MODEL_PATH = results_dst / "best_model.pt"

model = YOLO(str(MODEL_PATH))
net = model.model

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net = net.to(device).eval()

# -------------------------------------------------
# Params
# -------------------------------------------------
params = sum(p.numel() for p in net.parameters())
params_m = params / 1e6

# -------------------------------------------------
# GFLOPs
# -------------------------------------------------
dummy = torch.randn(1, 3, 832, 832).to(device)

macs, _ = profile(net, inputs=(dummy,), verbose=False)
gflops = macs / 1e9

# -------------------------------------------------
# Model size
# -------------------------------------------------
model_size_mb = MODEL_PATH.stat().st_size / (1024 ** 2)

# -------------------------------------------------
# Latency / FPS benchmark
# -------------------------------------------------
warmup = 20
runs = 100

with torch.no_grad():

    for _ in range(warmup):
        _ = net(dummy)

    torch.cuda.synchronize()
    start = time.time()

    for _ in range(runs):
        _ = net(dummy)

    torch.cuda.synchronize()
    end = time.time()

latency = (end - start) / runs
fps = 1 / latency

# -------------------------------------------------
# Print results
# -------------------------------------------------
print("\n📊 MODEL EFFICIENCY")
print(f"Params (M): {params_m:.2f}")
print(f"GFLOPs: {gflops:.2f}")
print(f"Model Size (MB): {model_size_mb:.2f}")
print(f"Latency (ms): {latency*1000:.2f}")
print(f"FPS: {fps:.2f}")

# -------------------------------------------------
# Save for paper
# -------------------------------------------------
efficiency = pd.DataFrame([{
    "model": "YOLOv11m",
    "params_M": round(params_m,3),
    "GFLOPs": round(gflops,3),
    "size_MB": round(model_size_mb,3),
    "latency_ms": round(latency*1000,3),
    "FPS": round(fps,3)
}])

efficiency_path = results_dst / "model_efficiency.csv"
efficiency.to_csv(efficiency_path, index=False)

print(f"\n📄 Efficiency table saved → {efficiency_path}")


📊 MODEL EFFICIENCY
Params (M): 2.59
GFLOPs: 5.45
Model Size (MB): 5.18
Latency (ms): 10.90
FPS: 91.76

📄 Efficiency table saved → /kaggle/working/yolo_results/model_efficiency.csv


MODEL OUTPUT

In [6]:
import os
import shutil
from pathlib import Path

runs_root = Path("/kaggle/working/runs")
save_dir = Path("/kaggle/working/yolo_results")
save_dir.mkdir(exist_ok=True)

candidate_dirs = []
if runs_root.exists():
    candidate_dirs.extend([path for path in runs_root.glob("*") if path.is_dir()])
    candidate_dirs.extend([path for path in runs_root.glob("detect/train*") if path.is_dir()])

candidate_dirs = sorted(set(candidate_dirs), key=os.path.getmtime)
latest_run = candidate_dirs[-1] if candidate_dirs else None

if latest_run is not None:
    print("Latest run found:", latest_run)
    shutil.copytree(latest_run, save_dir / "experiment", dirs_exist_ok=True)

    weights_dir = latest_run / "weights"
    if (weights_dir / "best.pt").exists():
        shutil.copy2(weights_dir / "best.pt", save_dir / "best_model.pt")
    if (weights_dir / "last.pt").exists():
        shutil.copy2(weights_dir / "last.pt", save_dir / "last_model.pt")
else:
    print("No run directory found under /kaggle/working/runs.")
    print("Using any existing files already present in /kaggle/working/yolo_results.")

saved_files = sorted(path.name for path in save_dir.iterdir()) if save_dir.exists() else []
if not saved_files:
    raise FileNotFoundError(
        "No model output files were found. Run the training cell first, then re-run this cell."
    )

print("Saved files:")
print(saved_files)

Latest run found: /kaggle/working/runs/weedmaize_yolo11n_feature_kd
Saved files:
['BoxF1_curve.png', 'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'best_model.pt', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'experiment', 'feature_kd_config.json', 'last_model.pt', 'model_efficiency.csv', 'results.csv', 'results.png']


In [7]:
# ============================================================
# STUDENT MODEL EVALUATION  — YOLO11n Feature Distillation
# Run this cell AFTER the KD training cell completes.
# Produces KD loss plots, test metrics, CSVs, JSON summary, and ZIP export.
# ============================================================

import json
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ultralytics import YOLO

BASE_DIR = Path("/kaggle/working/stratified_dataset")
YOLO_DIR = BASE_DIR / "weed_yolo"
YAML_PATH = YOLO_DIR / "dataset.yaml"
RESULTS_DIR = Path("/kaggle/working/yolo_results")
MODEL_PATH = RESULTS_DIR / "best_model.pt"
LAST_MODEL_PATH = RESULTS_DIR / "last_model.pt"
RESULTS_CSV = RESULTS_DIR / "results.csv"
KD_CONFIG_PATH = RESULTS_DIR / "feature_kd_config.json"
EXPORT_ROOT = Path("/kaggle/working/student_feature_kd_export")

for required_path in [YAML_PATH, MODEL_PATH, RESULTS_CSV]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file not found: {required_path}")

plots_dir = EXPORT_ROOT / "plots"
csv_dir = EXPORT_ROOT / "csv"
model_dir = EXPORT_ROOT / "models"
for directory in [plots_dir, csv_dir, model_dir]:
    directory.mkdir(parents=True, exist_ok=True)

shutil.copy2(MODEL_PATH, model_dir / "best.pt")
if LAST_MODEL_PATH.exists():
    shutil.copy2(LAST_MODEL_PATH, model_dir / "last.pt")
if KD_CONFIG_PATH.exists():
    shutil.copy2(KD_CONFIG_PATH, EXPORT_ROOT / "feature_kd_config.json")


def scol(frame, *names):
    lookup = {column.lower(): column for column in frame.columns}
    for name in names:
        key = name.lower()
        if key in lookup:
            return lookup[key]
    return None


# ──────────────────────────────────────────────────────────────
# 1. TRAINING METRICS + DISTILLATION LOSS CURVES
# ──────────────────────────────────────────────────────────────
df = pd.read_csv(RESULTS_CSV)
df.columns = df.columns.str.strip()
df.to_csv(csv_dir / "training_metrics.csv", index=False)

if "epoch" in df.columns:
    epochs = df["epoch"]
else:
    epochs = pd.Series(np.arange(1, len(df) + 1), name="epoch")

feat_kd_col = scol(df, "train/feat_kd_loss", "train_feat_kd_loss")
logit_kd_col = scol(df, "train/logit_kd_loss", "train_logit_kd_loss")
map50_col = scol(df, "metrics/mAP50(B)", "metrics/map50", "map50")
map95_col = scol(df, "metrics/mAP50-95(B)", "metrics/map50-95", "map50-95")
precision_col = scol(df, "metrics/precision(B)", "metrics/precision", "precision")
recall_col = scol(df, "metrics/recall(B)", "metrics/recall", "recall")

if feat_kd_col is None or logit_kd_col is None:
    raise RuntimeError(
        "Distillation loss columns were not found in results.csv. "
        "Re-run the KD training cell before running this evaluation cell."
    )

distill_history = pd.DataFrame(
    {
        "epoch": epochs,
        "feat_kd_loss": df[feat_kd_col],
        "logit_kd_loss": df[logit_kd_col],
        "total_distill_loss": df[feat_kd_col] + df[logit_kd_col],
    }
)
distill_history.to_csv(csv_dir / "distillation_loss_history.csv", index=False)

BLUE = "#2563EB"
RED = "#DC2626"
GREEN = "#16A34A"
ORANGE = "#EA580C"
SLATE = "#334155"

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("YOLO11n Feature Distillation Training Dashboard", fontsize=15, fontweight="bold")

# Detection losses
loss_axes = axes[0, 0]
for key, label, color in [
    (scol(df, "train/box_loss", "train_box_loss"), "Box", BLUE),
    (scol(df, "train/cls_loss", "train_cls_loss"), "Cls", RED),
    (scol(df, "train/dfl_loss", "train_dfl_loss"), "DFL", GREEN),
]:
    if key:
        loss_axes.plot(epochs, df[key], label=label, linewidth=2, color=color)
loss_axes.set_title("Detection Losses", fontweight="bold")
loss_axes.set_xlabel("Epoch")
loss_axes.set_ylabel("Loss")
loss_axes.grid(linestyle="--", alpha=0.35)
loss_axes.legend()

# Distillation losses
kd_axes = axes[0, 1]
kd_axes.plot(epochs, distill_history["feat_kd_loss"], label="Feature KD", linewidth=2, color=ORANGE)
kd_axes.plot(epochs, distill_history["logit_kd_loss"], label="Logit KD", linewidth=2, color=SLATE)
kd_axes.plot(epochs, distill_history["total_distill_loss"], label="Feature+Logit", linewidth=2, color="black")
kd_axes.set_title("Distillation Losses", fontweight="bold")
kd_axes.set_xlabel("Epoch")
kd_axes.set_ylabel("Loss")
kd_axes.grid(linestyle="--", alpha=0.35)
kd_axes.legend()

# mAP curves
map_axes = axes[1, 0]
if map50_col:
    map_axes.plot(epochs, df[map50_col], label="mAP@0.50", linewidth=2, color=GREEN)
if map95_col:
    map_axes.plot(epochs, df[map95_col], label="mAP@0.50:0.95", linewidth=2, color=BLUE)
map_axes.set_title("Validation mAP", fontweight="bold")
map_axes.set_xlabel("Epoch")
map_axes.set_ylabel("mAP")
map_axes.grid(linestyle="--", alpha=0.35)
map_axes.legend()

# Precision / Recall
pr_axes = axes[1, 1]
if precision_col:
    pr_axes.plot(epochs, df[precision_col], label="Precision", linewidth=2, color=BLUE)
if recall_col:
    pr_axes.plot(epochs, df[recall_col], label="Recall", linewidth=2, color=RED)
pr_axes.set_title("Precision / Recall", fontweight="bold")
pr_axes.set_xlabel("Epoch")
pr_axes.set_ylabel("Score")
pr_axes.grid(linestyle="--", alpha=0.35)
pr_axes.legend()

plt.tight_layout()
fig.savefig(plots_dir / "01_kd_training_dashboard.png", dpi=200, bbox_inches="tight")
plt.close()
print("[PLOT] 01_kd_training_dashboard.png")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(epochs, distill_history["feat_kd_loss"], label="Feature KD", linewidth=2.2, color=ORANGE)
ax.plot(epochs, distill_history["logit_kd_loss"], label="Logit KD", linewidth=2.2, color=SLATE)
ax.plot(epochs, distill_history["total_distill_loss"], label="Combined KD", linewidth=2.2, color="black")
best_kd_epoch = int(distill_history.loc[distill_history["total_distill_loss"].idxmin(), "epoch"])
best_kd_value = float(distill_history["total_distill_loss"].min())
ax.scatter([best_kd_epoch], [best_kd_value], color=RED, zorder=5)
ax.annotate(
    f"Lowest combined KD\nEpoch {best_kd_epoch}: {best_kd_value:.4f}",
    xy=(best_kd_epoch, best_kd_value),
    xytext=(10, 12),
    textcoords="offset points",
    arrowprops=dict(arrowstyle="->", color=RED),
)
ax.set_title("Distillation Loss Plot", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(linestyle="--", alpha=0.35)
ax.legend()
plt.tight_layout()
fig.savefig(plots_dir / "02_distillation_losses.png", dpi=200, bbox_inches="tight")
plt.close()
print("[PLOT] 02_distillation_losses.png")

# ──────────────────────────────────────────────────────────────
# 2. TEST-SET EVALUATION OF DISTILLED STUDENT
# ──────────────────────────────────────────────────────────────
student = YOLO(str(MODEL_PATH))
name_map = student.names if isinstance(student.names, dict) else dict(enumerate(student.names))
test_results = student.val(
    data=str(YAML_PATH),
    split="test",
    imgsz=640,
    batch=16,
    verbose=True,
    save_json=True,
    project=str(EXPORT_ROOT),
    name="test_eval",
)

box = test_results.box
speed = dict(test_results.speed)

rows = []
for class_id, class_name in name_map.items():
    precision = float(box.p[class_id]) if hasattr(box, "p") else 0.0
    recall = float(box.r[class_id]) if hasattr(box, "r") else 0.0
    map50 = float(box.ap50[class_id]) if hasattr(box, "ap50") else 0.0
    map95 = float(box.ap[class_id]) if hasattr(box, "ap") else 0.0
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    rows.append(
        {
            "class_id": class_id,
            "class_name": class_name,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "mAP50": map50,
            "mAP50_95": map95,
        }
    )

per_class_df = pd.DataFrame(rows)
per_class_df.to_csv(csv_dir / "test_per_class_metrics.csv", index=False)

overall = {
    "model": "yolo11n_feature_kd",
    "timestamp": datetime.now().isoformat(),
    "mAP50": float(box.map50),
    "mAP50_95": float(box.map),
    "precision": float(box.mp),
    "recall": float(box.mr),
    "f1": float(2 * box.mp * box.mr / (box.mp + box.mr + 1e-9)),
}
pd.DataFrame([overall]).to_csv(csv_dir / "test_overall_summary.csv", index=False)
pd.DataFrame([speed]).to_csv(csv_dir / "speed_metrics.csv", index=False)
print(f"[CSV] Overall test summary saved. mAP50-95 = {overall['mAP50_95']:.4f}")

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(per_class_df["class_name"], per_class_df["mAP50_95"], color=GREEN, alpha=0.85, edgecolor="white")
ax.set_title("Per-Class mAP@0.50:0.95 — Distilled Student", fontweight="bold")
ax.set_xlabel("Class")
ax.set_ylabel("mAP@0.50:0.95")
ax.set_ylim(0, 1.0)
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
fig.savefig(plots_dir / "03_per_class_map95.png", dpi=200, bbox_inches="tight")
plt.close()
print("[PLOT] 03_per_class_map95.png")

fig, ax = plt.subplots(figsize=(6, 4))
speed_labels = list(speed.keys())
speed_values = [speed[key] for key in speed_labels]
ax.bar(speed_labels, speed_values, color=[BLUE, GREEN, RED], alpha=0.85, edgecolor="white")
ax.set_title("Inference Speed per Image", fontweight="bold")
ax.set_ylabel("Milliseconds")
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
fig.savefig(plots_dir / "04_speed_metrics.png", dpi=200, bbox_inches="tight")
plt.close()
print("[PLOT] 04_speed_metrics.png")

# ──────────────────────────────────────────────────────────────
# 3. KD SUMMARY + EXPORT PACKAGE
# ──────────────────────────────────────────────────────────────
summary = {
    "student_model": str(MODEL_PATH),
    "dataset_yaml": str(YAML_PATH),
    "best_test_metrics": overall,
    "speed_ms": speed,
    "best_map50_epoch": int(epochs.iloc[df[map50_col].idxmax()]) if map50_col else None,
    "best_map50_95_epoch": int(epochs.iloc[df[map95_col].idxmax()]) if map95_col else None,
    "lowest_feature_kd": float(distill_history["feat_kd_loss"].min()),
    "lowest_logit_kd": float(distill_history["logit_kd_loss"].min()),
    "lowest_total_kd": best_kd_value,
    "lowest_total_kd_epoch": best_kd_epoch,
    "per_class_metrics": per_class_df.to_dict(orient="records"),
}
(EXPORT_ROOT / "student_kd_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("[JSON] student_kd_summary.json saved.")

readme = f"""# YOLO11n Feature Distillation Export
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Student
- Model: YOLO11n
- Teacher: YOLO11m best.pt
- KD method: FPN attention transfer + raw head logit matching

## Best Test Metrics
- mAP@0.50: {overall['mAP50']:.4f}
- mAP@0.50:0.95: {overall['mAP50_95']:.4f}
- Precision: {overall['precision']:.4f}
- Recall: {overall['recall']:.4f}
- F1: {overall['f1']:.4f}

## Distillation
- Lowest combined KD loss: {best_kd_value:.4f} at epoch {best_kd_epoch}
- Feature KD curve: plots/02_distillation_losses.png
- Full training dashboard: plots/01_kd_training_dashboard.png

## Files
- csv/training_metrics.csv
- csv/distillation_loss_history.csv
- csv/test_overall_summary.csv
- csv/test_per_class_metrics.csv
- student_kd_summary.json
- models/best.pt
"""
(EXPORT_ROOT / "README.md").write_text(readme, encoding="utf-8")

student_zip_root = "/kaggle/working/student_feature_kd_export"
compat_zip_root = "/kaggle/working/teacher_export_FINAL"
shutil.make_archive(student_zip_root, "zip", str(EXPORT_ROOT))
shutil.copyfile(f"{student_zip_root}.zip", f"{compat_zip_root}.zip")

print("\n============================================================")
print(f"✅ Student KD export ready → {student_zip_root}.zip")
print(f"✅ Compatibility ZIP      → {compat_zip_root}.zip")
print("============================================================\n")

try:
    from IPython.display import FileLink, display
    display(FileLink("student_feature_kd_export.zip"))
    display(FileLink("teacher_export_FINAL.zip"))
except Exception:
    print("Download the ZIPs from the Kaggle working directory.")

[PLOT] 01_kd_training_dashboard.png
[PLOT] 02_distillation_losses.png
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,584,297 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 115.0±37.3 MB/s, size: 765.9 KB)
val: Scanning /kaggle/working/stratified_dataset/weed_yolo/labels/test... 792 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 792/792 596.6it/s 1.3s
val: New cache created: /kaggle/working/stratified_dataset/weed_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 4.0it/s 12.5s
                   all        792      12826      0.763      0.687      0.762      0.474
                 CYPRO        534       1243      0.754      0.652      0.741      0.377
             CYPRO_max        577       1732      0.833      0.828      0.887        0.6
             CYPRO_min        535       1454   

/kaggle/working/student_feature_kd_export.zip

/kaggle/working/teacher_export_FINAL.zip

ONNX EXPORT

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ONNX EXPORT — Convert best.pt → best.onnx
# ═══════════════════════════════════════════════════════════════════════════
# Output: best.onnx saved to /kaggle/working/
# ═══════════════════════════════════════════════════════════════════════════

# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q ultralytics onnx onnxsim

import shutil
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
PT_INPUT  = Path("/kaggle/input/datasets/yas8agarwal/weed-detect/best.pt")
WORKING   = Path("/kaggle/working")
PT_COPY   = WORKING / "best.pt"
ONNX_OUT  = WORKING / "best.onnx"

# ── Validate source weights ───────────────────────────────────────────────────
if not PT_INPUT.exists():
    raise FileNotFoundError(
        f"❌ Weights not found at: {PT_INPUT}\n"
        "  → Make sure the dataset 'yas8agarwal/weed-detect' is attached\n"
        "    via: Add Data → yas8agarwal/weed-detect"
    )

print(f"✅ Source weights found : {PT_INPUT}")
print(f"   File size            : {PT_INPUT.stat().st_size / 1e6:.2f} MB")

# ── Copy weights to writable /kaggle/working/ ─────────────────────────────────
# Reason: Ultralytics saves .onnx alongside .pt; /kaggle/input/ is read-only
shutil.copy(PT_INPUT, PT_COPY)
print(f"✅ Weights copied to    : {PT_COPY}")

# ── Load model ────────────────────────────────────────────────────────────────
from ultralytics import YOLO

print("\n🔄 Loading YOLO model ...")
model = YOLO(str(PT_COPY))
print(f"   Model type  : {type(model.model).__name__}")
print(f"   Task        : {model.task}")

# ── Export to ONNX ────────────────────────────────────────────────────────────
print("\n🔄 Exporting to ONNX ...")
onnx_path = model.export(
    format   = "onnx",
    imgsz    = 832,      # must match training imgsz
    opset    = 17,       # opset 17: compatible with ORT >= 1.15, TRT 8.6+
    simplify = True,     # run onnx-simplifier for a cleaner graph
    dynamic  = False,    # static batch=1; set True for variable batch size
    half     = False,    # FP32; set True only if your runtime supports FP16
)

onnx_path = Path(onnx_path)

# ── Move .onnx to /kaggle/working/ if not already there ──────────────────────
if onnx_path.resolve() != ONNX_OUT.resolve():
    shutil.move(str(onnx_path), str(ONNX_OUT))
    onnx_path = ONNX_OUT

print(f"\n✅ ONNX model saved     : {onnx_path}")
print(f"   File size            : {onnx_path.stat().st_size / 1e6:.2f} MB")

# ── ONNX graph validation ─────────────────────────────────────────────────────
print("\n🔄 Validating ONNX graph ...")
try:
    import onnx
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    print("✅ ONNX graph validation passed.")

    # Print input / output shapes
    for inp in onnx_model.graph.input:
        shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
        print(f"   Input  : {inp.name}  →  {shape}")
    for out in onnx_model.graph.output:
        shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
        print(f"   Output : {out.name}  →  {shape}")

except ImportError:
    print("⚠️  onnx not installed — skipping validation. Run: pip install onnx")
except Exception as e:
    print(f"⚠️  Validation warning (model may still work): {e}")

# ── Optional: quick ONNX Runtime inference test ───────────────────────────────
print("\n🔄 Running ORT inference smoke-test ...")
try:
    import onnxruntime as ort
    import numpy as np

    sess    = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    inp_name = sess.get_inputs()[0].name
    dummy   = np.random.rand(1, 3, 832, 832).astype(np.float32)
    outputs = sess.run(None, {inp_name: dummy})

    print("✅ ORT smoke-test passed.")
    print(f"   Output shapes : {[o.shape for o in outputs]}")

except ImportError:
    print("⚠️  onnxruntime not installed — skipping ORT test. Run: pip install onnxruntime")
except Exception as e:
    print(f"⚠️  ORT smoke-test warning: {e}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  EXPORT SUMMARY")
print("═" * 60)
print(f"  Source  .pt  : {PT_INPUT}")
print(f"  Output  .onnx: {onnx_path}")
print(f"  Input shape  : [1, 3, 832, 832]")
print(f"  Opset        : 17")
print(f"  Simplified   : True")
print(f"  Precision    : FP32")
print("═" * 60)